# Compare ASCAT Legacy BUFR vs H121 CDR

Compares surface soil moisture from two ASCAT Metop-C products:

| | Legacy BUFR | H121 CDR |
|---|---|---|
| Format | BUFR (`.bfr`) orbit files | NetCDF hourly files |
| Product | ASCSMO02 — original 25 km swath | H SAF CDR — 12.5 km DGG |
| Files/day | ~14 orbit files | 24 hourly files |
| Obs/day (raw) | ~950k | ~5.6M |
| Obs/day (after QC) | ~60k | ~435k |

The ~7× obs count difference is expected: H121 is at 12.5 km resolution (vs ~25 km BUFR) and stores data on a fixed Discrete Global Grid, so it has ~4× more obs per unit area plus some repeated location coverage. Comparison is done on a 0.25° grid (mean per cell) to put both products on equal footing.

**QC applied** follows GEOSldas `clsm_ensupd_read_obs.F90`:
- SSM in [0, 100] % saturation
- Processing flag == 0 (H121 pre-filters this; all valid-SSM H121 obs have pf=0)
- Correction flag == 0 or 4
- Topographic complexity ≤ 10%
- Inundation/wetland fraction ≤ 10%
- Land fraction ≥ 0.9 omitted for BUFR — the `landFraction` field has inconsistent per-message replication structure that prevents reliable per-obs alignment with eccodes; ocean/coast obs are screened by the SSM [0,100] range check instead

**Note**: Legacy BUFR files are stored in `transfer.tar.gz` — extract before running:
```
cd /Users/amfox/Desktop/ASCAT_SSM_CDR/legacy_bufr/metop_c/2020/01
tar -xzf transfer.tar.gz
```

**Kernel**: `regrid` mamba env (eccodes 2.44.0)

In [ ]:
import numpy as np
import glob
import os
from datetime import datetime, timedelta

import eccodes as ecc
import netCDF4 as nc

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Legacy BUFR: extract transfer.tar.gz in this directory before running
BUFR_DIR = '/Users/amfox/Desktop/ASCAT_SSM_CDR/legacy_bufr/metop_c/2020/01'
H121_DIR = '/Users/amfox/Desktop/ASCAT_SSM_CDR/H121/metop_c/2020/01'

# Day to compare
DATE = datetime(2020, 1, 1)
DATE_STR = DATE.strftime('%Y%m%d')

# Grid resolution for binned comparison
GRID_RES = 0.25  # degrees

# Metop-C BUFR file prefix
BUFR_PREFIX = 'M03-ASCA-ASCSMO02-NA-5.0-'

## Reader functions

In [ ]:
def read_ascat_bufr_day(bufr_dir, date, prefix, printflag=False):
    """Read all legacy BUFR files for one day.

    Returns dict of arrays: lat, lon, ssm (% saturation), smpf, smcf, tpcx, iwfr.

    Notes:
    - landFraction (ALFR) skipped — inconsistent replication structure per message.
    - Some messages store QC fields as a single scalar for the whole message rather
      than per-obs. These are broadcast to match lat/lon length before appending.
    - Messages where any field length is neither 1 nor N_lat are skipped (rare).
    """
    date_str = date.strftime('%Y%m%d')
    pattern = os.path.join(bufr_dir, f'{prefix}{date_str}*.bfr')
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f'No BUFR files found matching {pattern}')
    print(f'Found {len(files)} BUFR files for {date_str}')

    KEYS = {
        'ssm':  'surfaceSoilMoisture',
        'smpf': 'soilMoistureProcessingFlag',
        'smcf': 'soilMoistureCorrectionFlag',
        'tpcx': 'topographicComplexity',
        'iwfr': 'inundationAndWetlandFraction',
        'lat':  'latitude',
        'lon':  'longitude',
    }

    out = {k: [] for k in KEYS}
    n_skip = 0

    for f in files:
        if printflag:
            print(f'  Reading {os.path.basename(f)}')
        with open(f, 'rb') as fh:
            bufr = ecc.codes_bufr_new_from_file(fh)
            while bufr is not None:
                ecc.codes_set(bufr, 'unpack', 1)

                arrays = {k: ecc.codes_get_array(bufr, v) for k, v in KEYS.items()}
                n = len(arrays['lat'])

                # skip if any field is neither scalar (1) nor per-obs (n)
                if any(len(a) not in (1, n) for a in arrays.values()):
                    n_skip += 1
                    ecc.codes_release(bufr)
                    bufr = ecc.codes_bufr_new_from_file(fh)
                    continue

                # broadcast scalar fields to per-obs length
                for k, arr in arrays.items():
                    if len(arr) == 1:
                        arrays[k] = np.repeat(arr, n)

                for k, arr in arrays.items():
                    out[k].extend(arr.tolist())

                ecc.codes_release(bufr)
                bufr = ecc.codes_bufr_new_from_file(fh)

    if n_skip:
        print(f'  Skipped {n_skip} messages with genuinely inconsistent field lengths')

    return {k: np.array(v) for k, v in out.items()}


def qc_bufr(d):
    """Apply GEOSldas QC to BUFR data dict (minus land fraction). Returns boolean mask."""
    mask = (
        (d['ssm']  >= 0.)  & (d['ssm']  <= 100.) &   # valid SSM range (also screens ocean)
        (d['smpf'] == 0)                          &   # processing flag clear
        ((d['smcf'] == 0) | (d['smcf'] == 4))    &   # correction flag 0 or 4
        (d['tpcx'] >= 0.)  & (d['tpcx'] <= 10.)  &   # topographic complexity <= 10%
        (d['iwfr'] >= 0.)  & (d['iwfr'] <= 10.)      # wetland fraction <= 10%
    )
    return mask

In [ ]:
def read_h121_day(h121_dir, date, printflag=False):
    """Read all H121 NetCDF files for one day.

    Returns dict with arrays: lat, lon, ssm (% saturation), pf, cf, wf, tc.
    netCDF4 auto-applies scale_factor; use .filled() to replace masked fill values.
    """
    date_str = date.strftime('%Y%m%d')
    # H121 filenames contain two timestamps; match on the obs-start date (second timestamp)
    pattern = os.path.join(h121_dir, f'*_{date_str}*.nc')
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f'No H121 files found matching {pattern}')
    print(f'Found {len(files)} H121 files for {date_str}')

    out = {k: [] for k in ['lat', 'lon', 'ssm', 'pf', 'cf', 'wf', 'tc']}

    for f in files:
        if printflag:
            print(f'  Reading {os.path.basename(f)}')
        ds = nc.Dataset(f)
        # netCDF4 auto-applies scale_factor when accessing variables;
        # .filled() replaces masked (fill value) positions with a sentinel
        lat = ds.variables['latitude'][:].filled(np.nan)           # degrees
        lon = ds.variables['longitude'][:].filled(np.nan)          # degrees
        ssm = ds.variables['surface_soil_moisture'][:].filled(np.nan)  # % saturation
        pf  = ds.variables['processing_flag'][:].filled(255)
        cf  = ds.variables['correction_flag'][:].filled(255)
        wf  = ds.variables['wetland_fraction'][:].filled(-128)
        tc  = ds.variables['topographic_complexity'][:].filled(-128)
        ds.close()

        out['lat'].extend(lat.tolist())
        out['lon'].extend(lon.tolist())
        out['ssm'].extend(ssm.tolist())
        out['pf'].extend(pf.tolist())
        out['cf'].extend(cf.tolist())
        out['wf'].extend(wf.tolist())
        out['tc'].extend(tc.tolist())

    return {k: np.array(v) for k, v in out.items()}


def qc_h121(d):
    """Apply GEOSldas-equivalent QC to H121 data dict. Returns boolean mask."""
    mask = (
        (d['ssm'] >= 0.)   & (d['ssm'] <= 100.)  &   # valid SSM range
        (d['pf']  == 0)                           &   # processing flag clear
        ((d['cf'] == 0) | (d['cf'] == 4))         &   # correction flag 0 or 4
        (d['wf']  >= 0.)   & (d['wf']  <= 10.)   &   # wetland fraction <= 10%
        (d['tc']  >= 0.)   & (d['tc']  <= 10.)       # topographic complexity <= 10%
        # note: H121 has no direct equivalent to BUFR land fraction (ALFR)
    )
    return mask

## Read data

In [ ]:
bufr_raw = read_ascat_bufr_day(BUFR_DIR, DATE, BUFR_PREFIX)
bufr_qc  = qc_bufr(bufr_raw)

bufr_lat = bufr_raw['lat'][bufr_qc]
bufr_lon = bufr_raw['lon'][bufr_qc]
bufr_ssm = bufr_raw['ssm'][bufr_qc]

print(f'BUFR  — raw: {len(bufr_raw["ssm"]):,}  QC pass: {bufr_qc.sum():,}')
print(f'BUFR SSM range: {bufr_ssm.min():.1f} – {bufr_ssm.max():.1f} %sat')

In [ ]:
h121_raw = read_h121_day(H121_DIR, DATE)
h121_qc  = qc_h121(h121_raw)

h121_lat = h121_raw['lat'][h121_qc]
h121_lon = h121_raw['lon'][h121_qc]
h121_ssm = h121_raw['ssm'][h121_qc]

print(f'H121  — raw: {len(h121_raw["ssm"]):,}  QC pass: {h121_qc.sum():,}')
print(f'H121 SSM range: {h121_ssm.min():.1f} – {h121_ssm.max():.1f} %sat')

## Grid to 0.25° for comparison

In [ ]:
def bin_to_grid(lat, lon, ssm, res=0.25):
    """Bin obs to regular lat/lon grid, return grid_lat, grid_lon, grid_ssm (mean per cell)."""
    ll_lat, ll_lon = -90., -180.
    n_lat = int(180. / res)
    n_lon = int(360. / res)
    
    i_lat = np.clip(np.floor((lat - ll_lat) / res).astype(int), 0, n_lat - 1)
    i_lon = np.clip(np.floor((lon - ll_lon) / res).astype(int), 0, n_lon - 1)
    
    ssm_sum   = np.full((n_lat, n_lon), 0.)
    ssm_count = np.full((n_lat, n_lon), 0)
    
    np.add.at(ssm_sum,   (i_lat, i_lon), ssm)
    np.add.at(ssm_count, (i_lat, i_lon), 1)
    
    with np.errstate(invalid='ignore'):
        ssm_mean = np.where(ssm_count > 0, ssm_sum / ssm_count, np.nan)
    
    grid_lat = ll_lat + (np.arange(n_lat) + 0.5) * res
    grid_lon = ll_lon + (np.arange(n_lon) + 0.5) * res
    
    return grid_lat, grid_lon, ssm_mean, ssm_count


glat, glon, bufr_grid, bufr_cnt = bin_to_grid(bufr_lat, bufr_lon, bufr_ssm, GRID_RES)
_, _,       h121_grid, h121_cnt = bin_to_grid(h121_lat, h121_lon, h121_ssm, GRID_RES)

# cells with obs in both
both = np.isfinite(bufr_grid) & np.isfinite(h121_grid)
print(f'Grid cells with obs in both: {both.sum():,}')
print(f'Grid cells BUFR only: {(np.isfinite(bufr_grid) & ~np.isfinite(h121_grid)).sum():,}')
print(f'Grid cells H121 only: {(np.isfinite(h121_grid) & ~np.isfinite(bufr_grid)).sum():,}')

## Statistics

In [ ]:
b = bufr_grid[both]
h = h121_grid[both]
diff = h - b

bias   = np.nanmean(diff)
rmsd   = np.sqrt(np.nanmean(diff**2))
r      = np.corrcoef(b, h)[0, 1]
ubrmsd = np.sqrt(rmsd**2 - bias**2)

print(f'Statistics (H121 - BUFR), {DATE_STR}, n={both.sum():,} grid cells')
print(f'  Bias:    {bias:+.3f} %sat')
print(f'  RMSD:    {rmsd:.3f} %sat')
print(f'  ubRMSD:  {ubrmsd:.3f} %sat')
print(f'  R:       {r:.4f}')

## Scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.hexbin(b, h, gridsize=60, cmap='viridis', mincnt=1)
ax.plot([0, 100], [0, 100], 'r--', lw=1, label='1:1')

ax.set_xlabel('BUFR SSM (% saturation)')
ax.set_ylabel('H121 SSM (% saturation)')
ax.set_title(f'ASCAT Metop-C SSM — {DATE_STR}\n0.25° gridded means')
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
ax.legend()

stats_txt = f'n={both.sum():,}\nbias={bias:+.2f}\nRMSD={rmsd:.2f}\nR={r:.3f}'
ax.text(0.05, 0.95, stats_txt, transform=ax.transAxes, va='top',
        fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
plt.show()

## Maps

In [ ]:
GLON2, GLAT2 = np.meshgrid(glon, glat)

def make_map_ax(ax, data, title, vmin=0, vmax=100, cmap='YlOrBr'):
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
    masked = np.ma.masked_invalid(data)
    sc = ax.pcolormesh(GLON2, GLAT2, masked, vmin=vmin, vmax=vmax, cmap=cmap,
                       transform=ccrs.PlateCarree(), zorder=1)
    ax.set_title(title, fontsize=10)
    return sc

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(3, 1, figsize=(14, 14),
                          subplot_kw={'projection': proj})

sc1 = make_map_ax(axes[0], bufr_grid, f'BUFR SSM — {DATE_STR}')
sc2 = make_map_ax(axes[1], h121_grid, f'H121 SSM — {DATE_STR}')

diff_grid = np.full_like(bufr_grid, np.nan)
diff_grid[both] = h121_grid[both] - bufr_grid[both]
sc3 = make_map_ax(axes[2], diff_grid, f'H121 − BUFR (% saturation)',
                  vmin=-20, vmax=20, cmap='RdBu_r')

for sc, ax in zip([sc1, sc2, sc3], axes):
    plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.02, pad=0.02)

plt.tight_layout()
plt.show()

## Obs counts map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4),
                          subplot_kw={'projection': proj})

for ax, cnt, label in zip(axes,
                          [bufr_cnt.astype(float), h121_cnt.astype(float)],
                          ['BUFR obs count', 'H121 obs count']):
    cnt_plot = cnt.copy()
    cnt_plot[cnt_plot == 0] = np.nan
    sc = make_map_ax(ax, cnt_plot, f'{label} — {DATE_STR}',
                     vmin=1, vmax=20, cmap='plasma')
    plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.025, pad=0.02)

plt.tight_layout()
plt.show()

## Distribution comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bins = np.arange(0, 101, 2)
ax.hist(bufr_ssm, bins=bins, density=True, alpha=0.6, label=f'BUFR (n={len(bufr_ssm):,})', color='steelblue')
ax.hist(h121_ssm, bins=bins, density=True, alpha=0.6, label=f'H121 (n={len(h121_ssm):,})', color='darkorange')

ax.set_xlabel('SSM (% saturation)')
ax.set_ylabel('Density')
ax.set_title(f'SSM distribution — {DATE_STR} (after QC, point obs)')
ax.legend()
plt.tight_layout()
plt.show()

---
## GEOSldas Observation Space

Diagnostic `ObsFcstAna` output from a `hsaf_cdr_test_DAv8_M36` run for 2020-01-01. Both
legacy ASCAT (species 9–11, Metop A/B/C) and H121 CDR (species 15–17) were ingested
simultaneously in **monitor-only mode** — no assimilation, `assim_flag=0` throughout.

This lets us check:
1. Whether the H121 GEOSldas reader produces a plausible number and distribution of obs
2. Whether innovations (O−F) are comparable between the two products — a large systematic
   difference would indicate a unit mismatch or calibration issue in the new reader
3. Spatial patterns in both obs and innovations

*Obs values are stored as degree of saturation (0–1) after GEOSldas rescales from the native
% saturation units. `obsparam_errstd=9.0` reflects the original % units before rescaling.*

In [ ]:
OFA_BASE   = '/Users/amfox/Desktop/geosldas-analysis/data/hsaf_cdr_test'
OFA_SUFFIX = 'output/SMAP_EASEv2_M36_GLOBAL/ana/ens_avg/Y2020/M01'

EXPTS = {
    'monitor':        'hsaf_cdr_test_DAv8_M36',
    'scale':          'hsaf_cdr_test_DAv8_M36_scale',
    'eumetsat_assim': 'hsaf_cdr_test_DAv8_M36_EUMETSAT_assim',
    'hsaf_assim':     'hsaf_cdr_test_DAv8_M36_HSAF_assim',
}

def ofa_dir(key):
    return f'{OFA_BASE}/{EXPTS[key]}/{OFA_SUFFIX}'

# Species metadata
SPECIES = {
    9:  {'name': 'ASCAT_META_SM',      'product': 'Legacy', 'platform': 'Metop-A'},
    10: {'name': 'ASCAT_METB_SM',      'product': 'Legacy', 'platform': 'Metop-B'},
    11: {'name': 'ASCAT_METC_SM',      'product': 'Legacy', 'platform': 'Metop-C'},
    15: {'name': 'ASCAT_HSAF_META_SM', 'product': 'H121',   'platform': 'Metop-A'},
    16: {'name': 'ASCAT_HSAF_METB_SM', 'product': 'H121',   'platform': 'Metop-B'},
    17: {'name': 'ASCAT_HSAF_METC_SM', 'product': 'H121',   'platform': 'Metop-C'},
}
ASCAT_SPECIES = list(SPECIES.keys())
legacy_ids = [9, 10, 11]
h121_ids   = [15, 16, 17]


def read_ofa_files(dir_path):
    """Load all ObsFcstAna nc4 files from a directory.

    Returns dict keyed by species id, each holding arrays:
      lat, lon, obs, fcst, ana, innov, incr, assim_flag, time_tag (str)
    """
    files = sorted(glob.glob(os.path.join(dir_path, '*.nc4')))
    print(f'Reading {len(files)} ObsFcstAna files from {dir_path}')

    data = {s: {'lat': [], 'lon': [], 'obs': [], 'fcst': [], 'ana': [],
                'innov': [], 'incr': [], 'assim': [], 'time_tag': []}
            for s in ASCAT_SPECIES}

    for f in files:
        tag = os.path.basename(f).split('.')[-2]
        ds = nc.Dataset(f)
        sp  = ds.variables['species'][:]
        obs = ds.variables['obs'][:]
        fcs = ds.variables['fcst'][:]
        ana = ds.variables['ana'][:]
        lat = ds.variables['lat'][:]
        lon = ds.variables['lon'][:]
        af  = ds.variables['assim_flag'][:]
        ds.close()

        for s in ASCAT_SPECIES:
            m = sp == s
            if m.sum() == 0:
                continue
            data[s]['lat'].extend(lat[m].tolist())
            data[s]['lon'].extend(lon[m].tolist())
            data[s]['obs'].extend(obs[m].tolist())
            data[s]['fcst'].extend(fcs[m].tolist())
            data[s]['ana'].extend(ana[m].tolist())
            data[s]['innov'].extend((obs[m] - fcs[m]).tolist())
            data[s]['incr'].extend((ana[m]  - fcs[m]).tolist())
            data[s]['assim'].extend(af[m].tolist())
            data[s]['time_tag'].extend([tag] * m.sum())

    return {s: {k: np.array(v) for k, v in d.items()} for s, d in data.items()}


ofa = read_ofa_files(ofa_dir('monitor'))

In [ ]:
# ── Per-platform summary statistics ─────────────────────────────────────────
header = f"{'Product':<8} {'Platform':<9} {'N':>7}  {'Obs mean':>9} {'Obs std':>8}  {'Fcst mean':>9} {'Fcst std':>8}  {'O-F bias':>9} {'O-F std':>8} {'O-F RMSD':>9}  {'Assim%':>7}"
print(header)
print('-' * len(header))

for s in ASCAT_SPECIES:
    d = ofa[s]
    if len(d['obs']) == 0:
        continue
    obs   = d['obs']
    fcst  = d['fcst']
    innov = d['innov']
    n_assim = (d['assim'] == 1).sum()
    prod  = SPECIES[s]['product']
    plat  = SPECIES[s]['platform']

    bias  = innov.mean()
    std   = innov.std()
    rmsd  = np.sqrt((innov**2).mean())
    assim_pct = 100.0 * n_assim / len(obs)

    print(f"{prod:<8} {plat:<9} {len(obs):>7}  {obs.mean():>9.4f} {obs.std():>8.4f}  "
          f"{fcst.mean():>9.4f} {fcst.std():>8.4f}  "
          f"{bias:>+9.4f} {std:>8.4f} {rmsd:>9.4f}  {assim_pct:>7.1f}")

    if s in (11, 17):
        print()

print()
print("Units: degree of saturation (0–1)")

In [ ]:
# ── Per-platform × per-window statistics ─────────────────────────────────────
TIME_WINDOWS = sorted({tag for s in ASCAT_SPECIES for tag in ofa[s]['time_tag']})
legacy_ids = [9, 10, 11]
h121_ids   = [15, 16, 17]

def window_platform_table(ids, product_label):
    plat_ids = {SPECIES[s]['platform']: s for s in ids}
    platforms_order = ['Metop-A', 'Metop-B', 'Metop-C']

    def fmt_cell(n, obs_, fcst_, inn_):
        return (f"{n:>6d} {obs_.mean():.3f} {fcst_.mean():.3f}"
                f" {inn_.mean():+.3f} {inn_.std():.3f}")

    col_w = 34
    hdr = f"{'Window':<10}" + "".join(f"{p:^{col_w}}" for p in platforms_order + ['All'])
    sub = f"{'':10}" + "".join(f"{'N / obs / fcst / bias / std':^{col_w}}"
                               for _ in platforms_order + ['All'])
    sep = '-' * len(hdr)

    print(f"\n{product_label}")
    print(hdr); print(sub); print(sep)

    for tw in TIME_WINDOWS + ['Total']:
        row = f"{tw.split('_')[-1] if tw != 'Total' else 'Total':<10}"
        all_obs, all_fcst, all_inn = [], [], []
        for p in platforms_order:
            s = plat_ids[p]
            mask = slice(None) if tw == 'Total' else (ofa[s]['time_tag'] == tw)
            o, f_, i = ofa[s]['obs'][mask], ofa[s]['fcst'][mask], ofa[s]['innov'][mask]
            all_obs.append(o); all_fcst.append(f_); all_inn.append(i)
            row += f"{'—':^{col_w}}" if len(o) == 0 else f"{fmt_cell(len(o),o,f_,i):^{col_w}}"
        oc = np.concatenate(all_obs)
        fc = np.concatenate(all_fcst)
        ic = np.concatenate(all_inn)
        row += f"{fmt_cell(len(oc),oc,fc,ic):^{col_w}}"
        print(row)
        if tw == TIME_WINDOWS[-1]:
            print(sep)

    print("  Columns: N  /  obs mean  /  fcst mean  /  O-F bias  /  O-F std"
          "   (degree of saturation, 0–1)")

window_platform_table(legacy_ids, "Legacy ASCAT")
window_platform_table(h121_ids,   "H121 CDR")

In [ ]:
# ── Temporal obs counts + O-F innovation distributions ──────────────────────
TIME_WINDOWS = sorted({tag for s in ASCAT_SPECIES for tag in ofa[s]['time_tag']})
legacy_ids = [9, 10, 11]
h121_ids   = [15, 16, 17]

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# -- Left: total obs per 3h window, Legacy vs H121 ----------------------------
ax = axes[0]
x = np.arange(len(TIME_WINDOWS))
width = 0.35

legacy_counts = np.array([sum((ofa[s]['time_tag'] == tw).sum() for s in legacy_ids)
                           for tw in TIME_WINDOWS])
h121_counts   = np.array([sum((ofa[s]['time_tag'] == tw).sum() for s in h121_ids)
                           for tw in TIME_WINDOWS])

ax.bar(x - width/2, legacy_counts, width, color='steelblue',   label='Legacy (all platforms)')
ax.bar(x + width/2, h121_counts,   width, color='darkorange',  label='H121 (all platforms)')

ax.set_xticks(x)
ax.set_xticklabels([tw.split('_')[1] for tw in TIME_WINDOWS], rotation=45, ha='right', fontsize=7)
ax.set_xlabel('3h assimilation window')
ax.set_ylabel('Obs count (all platforms)')
ax.set_title('Total obs count per window')
ax.legend(fontsize=8)

# -- Centre: O-F histogram, Metop-C only, legacy vs H121 ----------------------
ax = axes[1]
bins = np.linspace(-0.8, 0.8, 80)
for s, label, color in [(11, 'Legacy Metop-C', 'steelblue'),
                         (17, 'H121 Metop-C',  'darkorange')]:
    inn = ofa[s]['innov']
    ax.hist(inn, bins=bins, density=True, alpha=0.7,
            label=f'{label}  bias={inn.mean():+.3f}, std={inn.std():.3f}', color=color)
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('O−F innovation (model space, 0–1)')
ax.set_ylabel('Density')
ax.set_title('Innovation distribution — Metop-C')
ax.legend(fontsize=8)

# -- Right: O-F histogram, all platforms, legacy vs H121 ----------------------
ax = axes[2]
for ids, label, color in [(legacy_ids, 'Legacy (all)', 'steelblue'),
                           (h121_ids,   'H121 (all)',   'darkorange')]:
    inn = np.concatenate([ofa[s]['innov'] for s in ids])
    ax.hist(inn, bins=bins, density=True, alpha=0.7,
            label=f'{label}  bias={inn.mean():+.3f}, std={inn.std():.3f}', color=color)
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('O−F innovation (model space, 0–1)')
ax.set_ylabel('Density')
ax.set_title('Innovation distribution — all platforms')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Maps: obs values on 1° grid ──────────────────────────────────────────────
MAP_RES = 1.0

def ofa_to_grid(lat, lon, values, res=MAP_RES):
    _, _, g, c = bin_to_grid(lat, lon, values, res=res)
    return g, c

ofa_obs_grid = {}
ofa_inn_grid = {}
for s in ASCAT_SPECIES:
    d = ofa[s]
    ofa_obs_grid[s], _ = ofa_to_grid(d['lat'], d['lon'], d['obs'])
    ofa_inn_grid[s], _ = ofa_to_grid(d['lat'], d['lon'], d['innov'])

n_lat_m = int(180. / MAP_RES)
n_lon_m = int(360. / MAP_RES)
glat_m  = -90.  + (np.arange(n_lat_m) + 0.5) * MAP_RES
glon_m  = -180. + (np.arange(n_lon_m) + 0.5) * MAP_RES
GLON_M, GLAT_M = np.meshgrid(glon_m, glat_m)

def map_ax_ofa(ax, data, title, vmin, vmax, cmap):
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
    masked = np.ma.masked_invalid(data)
    sc = ax.pcolormesh(GLON_M, GLAT_M, masked, vmin=vmin, vmax=vmax, cmap=cmap,
                       transform=ccrs.PlateCarree(), zorder=1)
    ax.set_title(title, fontsize=9)
    return sc

platforms = ['Metop-A', 'Metop-B', 'Metop-C']
legacy_sp = {SPECIES[s]['platform']: s for s in legacy_ids}
h121_sp   = {SPECIES[s]['platform']: s for s in h121_ids}

fig, axes = plt.subplots(2, 3, figsize=(18, 7), subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle(f'GEOSldas obs (degree of saturation, 0–1) — {DATE_STR}', fontsize=11)
fig.subplots_adjust(right=0.88, hspace=0.15, wspace=0.05)

for col, plat in enumerate(platforms):
    for row, (sp_dict, label) in enumerate([(legacy_sp, 'Legacy'), (h121_sp, 'H121')]):
        s = sp_dict[plat]
        sc = map_ax_ofa(axes[row, col], ofa_obs_grid[s], '',
                        vmin=0, vmax=1, cmap='YlOrBr')
        axes[row, col].set_title(f'{label} {plat} (n={len(ofa[s]["obs"]):,})', fontsize=9)

cax = fig.add_axes([0.90, 0.12, 0.015, 0.75])
fig.colorbar(sc, cax=cax, label='degree of saturation')
plt.show()

In [ ]:
# ── Innovation maps: 2 rows (Legacy, H121) × 3 cols (Metop A, B, C) ─────────
fig, axes = plt.subplots(2, 3, figsize=(18, 7), subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle(f'GEOSldas O−F innovations (degree of saturation) — {DATE_STR}', fontsize=11)
fig.subplots_adjust(right=0.88, hspace=0.15, wspace=0.05)

for col, plat in enumerate(platforms):
    for row, (sp_dict, label) in enumerate([(legacy_sp, 'Legacy'), (h121_sp, 'H121')]):
        s = sp_dict[plat]
        inn = ofa[s]['innov']
        sc = map_ax_ofa(axes[row, col], ofa_inn_grid[s], '',
                        vmin=-0.4, vmax=0.4, cmap='RdBu_r')
        axes[row, col].set_title(
            f'{label} {plat}  bias={inn.mean():+.3f}, std={inn.std():.3f}', fontsize=9)

cax = fig.add_axes([0.90, 0.12, 0.015, 0.75])
fig.colorbar(sc, cax=cax, label='O−F innovation (deg. sat.)')
plt.show()

---
## Gridded innovation comparison: Legacy vs H121

CDF-matching (scaling) is applied on a 0.25° lat/lon grid, combining all platforms. To
assess whether Legacy and H121 will receive similar corrections, we bin the all-platform
innovations onto that grid and compare spatially and via scatter plot.

In [ ]:
# ── Bin all-platform innovations onto 0.25° grid ────────────────────────────
INN_RES = 0.25

legacy_lat  = np.concatenate([ofa[s]['lat']   for s in legacy_ids])
legacy_lon  = np.concatenate([ofa[s]['lon']   for s in legacy_ids])
legacy_inn  = np.concatenate([ofa[s]['innov'] for s in legacy_ids])

h121_lat    = np.concatenate([ofa[s]['lat']   for s in h121_ids])
h121_lon    = np.concatenate([ofa[s]['lon']   for s in h121_ids])
h121_inn    = np.concatenate([ofa[s]['innov'] for s in h121_ids])

inn_glat, inn_glon, legacy_inn_grid, _ = bin_to_grid(legacy_lat, legacy_lon, legacy_inn, INN_RES)
_,         _,        h121_inn_grid,   _ = bin_to_grid(h121_lat,   h121_lon,   h121_inn,   INN_RES)

INN_GLON, INN_GLAT = np.meshgrid(inn_glon, inn_glat)

both_inn = np.isfinite(legacy_inn_grid) & np.isfinite(h121_inn_grid)
print(f'0.25° cells with innovations in both products: {both_inn.sum():,}')
print(f'Legacy only: {(np.isfinite(legacy_inn_grid) & ~np.isfinite(h121_inn_grid)).sum():,}')
print(f'H121 only:   {(np.isfinite(h121_inn_grid)   & ~np.isfinite(legacy_inn_grid)).sum():,}')

# ── Maps: Legacy | H121 | difference ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                          subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle(f'0.25° gridded O−F innovations (all platforms) — {DATE_STR}', fontsize=11)

diff_inn = np.full_like(legacy_inn_grid, np.nan)
diff_inn[both_inn] = h121_inn_grid[both_inn] - legacy_inn_grid[both_inn]

plots = [
    (legacy_inn_grid, 'Legacy ASCAT — all platforms', -0.4, 0.4, 'RdBu_r'),
    (h121_inn_grid,   'H121 CDR — all platforms',     -0.4, 0.4, 'RdBu_r'),
    (diff_inn,        'H121 − Legacy innovation',      -0.2, 0.2, 'PuOr_r'),
]
for ax, (data, title, vmin, vmax, cmap) in zip(axes, plots):
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
    sc = ax.pcolormesh(INN_GLON, INN_GLAT, np.ma.masked_invalid(data),
                       vmin=vmin, vmax=vmax, cmap=cmap,
                       transform=ccrs.PlateCarree(), zorder=1)
    ax.set_title(title, fontsize=10)
    plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.02, pad=0.02,
                 label='O−F (deg. sat.)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plot: gridded Legacy vs H121 innovations ────────────────────────
li = legacy_inn_grid[both_inn]
hi = h121_inn_grid[both_inn]
inn_diff = hi - li

bias_i  = inn_diff.mean()
rmsd_i  = np.sqrt((inn_diff**2).mean())
r_i     = np.corrcoef(li, hi)[0, 1]

fig, ax = plt.subplots(figsize=(6, 6))
ax.hexbin(li, hi, gridsize=60, cmap='viridis', mincnt=1, extent=[-0.6, 0.6, -0.6, 0.6])
lim = 0.6
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='1:1')
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('Legacy O−F innovation (deg. sat.)')
ax.set_ylabel('H121 O−F innovation (deg. sat.)')
ax.set_title(f'0.25° gridded innovations — {DATE_STR}\nall platforms combined')
ax.legend(fontsize=9)

stats_txt = f'n={both_inn.sum():,}\nbias={bias_i:+.4f}\nRMSD={rmsd_i:.4f}\nR={r_i:.4f}'
ax.text(0.05, 0.95, stats_txt, transform=ax.transAxes, va='top',
        fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
plt.show()

---
## Scaling and assimilation experiments

Four GEOSldas runs on 2020-01-01, all ingesting Legacy (species 9–11) + H121 (species 15–17):

| Key | Description |
|---|---|
| `monitor` | No scaling, no assimilation — raw obs/fcst comparison |
| `scale` | CDF-matching applied, monitor only — obs brought into model space |
| `eumetsat_assim` | CDF-matching + Legacy ASCAT assimilated; H121 monitored |
| `hsaf_assim` | CDF-matching + H121 assimilated; Legacy monitored |

In [ ]:
# ── Load all four experiments ────────────────────────────────────────────────
ofa_all = {key: read_ofa_files(ofa_dir(key)) for key in EXPTS}

### Effect of CDF-matching on obs values and innovations

In [ ]:
# ── Obs distribution before/after scaling, and innovation distributions ──────
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f'Effect of CDF-matching — {DATE_STR}', fontsize=11)

bins_obs  = np.linspace(0, 0.6, 80)
bins_inn  = np.linspace(-0.4, 0.4, 80)
fcst_all_leg = np.concatenate([ofa_all['monitor'][s]['fcst'] for s in legacy_ids])
fcst_all_h121 = np.concatenate([ofa_all['monitor'][s]['fcst'] for s in h121_ids])

for row, (ids, prod_label, color) in enumerate([
        (legacy_ids, 'Legacy ASCAT', 'steelblue'),
        (h121_ids,   'H121 CDR',     'darkorange')]):

    obs_raw   = np.concatenate([ofa_all['monitor'][s]['obs']   for s in ids])
    obs_scl   = np.concatenate([ofa_all['scale'][s]['obs']     for s in ids])
    fcst_scl  = np.concatenate([ofa_all['scale'][s]['fcst']    for s in ids])
    inn_raw   = np.concatenate([ofa_all['monitor'][s]['innov'] for s in ids])
    inn_scl   = np.concatenate([ofa_all['scale'][s]['innov']   for s in ids])

    # Left: obs distributions before/after scaling + fcst
    ax = axes[row, 0]
    ax.hist(obs_raw, bins=bins_obs, density=True, alpha=0.6,
            color=color, label=f'obs unscaled  mean={obs_raw.mean():.3f}')
    ax.hist(obs_scl, bins=bins_obs, density=True, alpha=0.6,
            color='gray', label=f'obs scaled    mean={obs_scl.mean():.3f}')
    ax.hist(fcst_scl, bins=bins_obs, density=True, alpha=0.4,
            color='k', label=f'fcst          mean={fcst_scl.mean():.3f}')
    ax.set_xlabel('deg. sat. (0–1)')
    ax.set_ylabel('Density')
    ax.set_title(f'{prod_label} — obs before/after scaling')
    ax.legend(fontsize=8)

    # Right: innovation distributions before/after scaling
    ax = axes[row, 1]
    ax.hist(inn_raw, bins=bins_inn, density=True, alpha=0.6,
            color=color, label=f'unscaled  bias={inn_raw.mean():+.3f} std={inn_raw.std():.3f}')
    ax.hist(inn_scl, bins=bins_inn, density=True, alpha=0.6,
            color='gray', label=f'scaled    bias={inn_scl.mean():+.3f} std={inn_scl.std():.3f}')
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('O−F innovation (deg. sat.)')
    ax.set_ylabel('Density')
    ax.set_title(f'{prod_label} — innovations before/after scaling')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Post-scaling gridded innovation comparison

In [ ]:
# ── Gridded post-scaling innovations: Legacy vs H121 ─────────────────────────
scl = ofa_all['scale']

scl_leg_lat  = np.concatenate([scl[s]['lat']   for s in legacy_ids])
scl_leg_lon  = np.concatenate([scl[s]['lon']   for s in legacy_ids])
scl_leg_inn  = np.concatenate([scl[s]['innov'] for s in legacy_ids])
scl_h121_lat = np.concatenate([scl[s]['lat']   for s in h121_ids])
scl_h121_lon = np.concatenate([scl[s]['lon']   for s in h121_ids])
scl_h121_inn = np.concatenate([scl[s]['innov'] for s in h121_ids])

_, _, scl_leg_grid,  _ = bin_to_grid(scl_leg_lat,  scl_leg_lon,  scl_leg_inn,  0.25)
_, _, scl_h121_grid, _ = bin_to_grid(scl_h121_lat, scl_h121_lon, scl_h121_inn, 0.25)

both_scl = np.isfinite(scl_leg_grid) & np.isfinite(scl_h121_grid)
li_scl = scl_leg_grid[both_scl]
hi_scl = scl_h121_grid[both_scl]
diff_scl = hi_scl - li_scl

bias_scl = diff_scl.mean()
rmsd_scl = np.sqrt((diff_scl**2).mean())
r_scl    = np.corrcoef(li_scl, hi_scl)[0, 1]

print(f'Post-scaling gridded innovations: n={both_scl.sum():,} cells')
print(f'  Legacy   bias={scl_leg_inn.mean():+.4f}  std={scl_leg_inn.std():.4f}')
print(f'  H121     bias={scl_h121_inn.mean():+.4f}  std={scl_h121_inn.std():.4f}')
print(f'  H121−Legacy  bias={bias_scl:+.4f}  RMSD={rmsd_scl:.4f}  R={r_scl:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
ax = axes[0]
lim = 0.15
ax.hexbin(li_scl, hi_scl, gridsize=60, cmap='viridis', mincnt=1,
          extent=[-lim, lim, -lim, lim])
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='1:1')
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('Legacy O−F (scaled, deg. sat.)')
ax.set_ylabel('H121 O−F (scaled, deg. sat.)')
ax.set_title(f'Post-scaling gridded innovations — {DATE_STR}')
ax.legend(fontsize=9)
stats_txt = f'n={both_scl.sum():,}\nbias={bias_scl:+.4f}\nRMSD={rmsd_scl:.4f}\nR={r_scl:.4f}'
ax.text(0.05, 0.95, stats_txt, transform=ax.transAxes, va='top',
        fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# Innovation histogram comparison: unscaled vs scaled, both products
ax = axes[1]
bins_i = np.linspace(-0.4, 0.4, 80)
leg_inn_raw = np.concatenate([ofa_all['monitor'][s]['innov'] for s in legacy_ids])
h121_inn_raw = np.concatenate([ofa_all['monitor'][s]['innov'] for s in h121_ids])
ax.hist(leg_inn_raw,  bins=bins_i, density=True, alpha=0.4, color='steelblue',
        label=f'Legacy unscaled  std={leg_inn_raw.std():.3f}')
ax.hist(h121_inn_raw, bins=bins_i, density=True, alpha=0.4, color='darkorange',
        label=f'H121 unscaled    std={h121_inn_raw.std():.3f}')
ax.hist(scl_leg_inn,  bins=bins_i, density=True, alpha=0.7, color='steelblue',
        histtype='step', lw=1.5, label=f'Legacy scaled    std={scl_leg_inn.std():.3f}')
ax.hist(scl_h121_inn, bins=bins_i, density=True, alpha=0.7, color='darkorange',
        histtype='step', lw=1.5, label=f'H121 scaled      std={scl_h121_inn.std():.3f}')
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('O−F innovation (deg. sat.)')
ax.set_ylabel('Density')
ax.set_title('Innovation distributions: before (fill) and after (outline) scaling')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Assimilation comparison: Legacy (EUMETSAT) vs H121 (HSAF)

In [ ]:
# ── Per-platform summary: innovations + analysis increments ─────────────────
for expt_key, label in [('eumetsat_assim', 'EUMETSAT_assim (Legacy assimilated)'),
                         ('hsaf_assim',     'HSAF_assim     (H121 assimilated)')]:
    d_all = ofa_all[expt_key]
    print(f'\n{label}')
    hdr = f"  {'Prod':<7} {'Plat':<9} {'N':>6}  {'O-F bias':>9} {'O-F std':>8}  {'A-F incr':>9} {'Incr std':>8}  {'Assim%':>7}"
    print(hdr)
    print('  ' + '-' * (len(hdr)-2))
    for s in ASCAT_SPECIES:
        d = d_all[s]
        if len(d['obs']) == 0:
            continue
        inn   = d['innov']
        incr  = d['incr']
        ap    = 100.0 * (d['assim'] == 1).sum() / len(d['obs'])
        prod  = SPECIES[s]['product']
        plat  = SPECIES[s]['platform']
        print(f"  {prod:<7} {plat:<9} {len(d['obs']):>6}  "
              f"{inn.mean():>+9.4f} {inn.std():>8.4f}  "
              f"{incr.mean():>+9.4f} {incr.std():>8.4f}  {ap:>7.1f}")
        if s in (11, 17):
            print()
print("  Units: degree of saturation (0–1)")

In [ ]:
# ── Analysis increment maps: EUMETSAT_assim vs HSAF_assim ────────────────────
# Combine all platforms; only obs with assim_flag=1 have a meaningful increment
def grid_assim_incr(expt_key, ids, res=0.25):
    d_all = ofa_all[expt_key]
    lat_list, lon_list, incr_list = [], [], []
    for s in ids:
        d = d_all[s]
        mask = d['assim'] == 1
        if mask.sum() == 0:
            continue
        lat_list.append(d['lat'][mask])
        lon_list.append(d['lon'][mask])
        incr_list.append(d['incr'][mask])
    if not lat_list:
        return np.full((int(180/res), int(360/res)), np.nan)
    lat_c  = np.concatenate(lat_list)
    lon_c  = np.concatenate(lon_list)
    incr_c = np.concatenate(incr_list)
    _, _, g, _ = bin_to_grid(lat_c, lon_c, incr_c, res)
    return g

incr_eumetsat = grid_assim_incr('eumetsat_assim', legacy_ids)
incr_hsaf     = grid_assim_incr('hsaf_assim',     h121_ids)

n_lat_g = int(180. / 0.25)
n_lon_g = int(360. / 0.25)
glat_g  = -90.  + (np.arange(n_lat_g) + 0.5) * 0.25
glon_g  = -180. + (np.arange(n_lon_g) + 0.5) * 0.25
GLON_G, GLAT_G = np.meshgrid(glon_g, glat_g)

both_incr = np.isfinite(incr_eumetsat) & np.isfinite(incr_hsaf)
diff_incr = np.full_like(incr_eumetsat, np.nan)
diff_incr[both_incr] = incr_hsaf[both_incr] - incr_eumetsat[both_incr]

fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                          subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle(f'0.25° gridded analysis increments (A−F, assimilated obs only) — {DATE_STR}',
             fontsize=11)

plots = [
    (incr_eumetsat, 'Legacy ASCAT (EUMETSAT_assim)',  -0.05, 0.05, 'RdBu_r'),
    (incr_hsaf,     'H121 CDR (HSAF_assim)',           -0.05, 0.05, 'RdBu_r'),
    (diff_incr,     'H121 − Legacy increment',         -0.03, 0.03, 'PuOr_r'),
]
for ax, (data, title, vmin, vmax, cmap) in zip(axes, plots):
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
    sc = ax.pcolormesh(GLON_G, GLAT_G, np.ma.masked_invalid(data),
                       vmin=vmin, vmax=vmax, cmap=cmap,
                       transform=ccrs.PlateCarree(), zorder=1)
    ax.set_title(title, fontsize=10)
    plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.02, pad=0.02,
                 label='A−F (deg. sat.)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: gridded analysis increments Legacy vs H121 ──────────────────────
li_incr = incr_eumetsat[both_incr]
hi_incr = incr_hsaf[both_incr]
diff_incr_vals = hi_incr - li_incr

bias_incr_s = diff_incr_vals.mean()
rmsd_incr_s = np.sqrt((diff_incr_vals**2).mean())
r_incr_s    = np.corrcoef(li_incr, hi_incr)[0, 1]

print(f'Gridded increment comparison: n={both_incr.sum():,} cells')
print(f'  Legacy incr   mean={li_incr.mean():+.5f}  std={li_incr.std():.5f}')
print(f'  H121 incr     mean={hi_incr.mean():+.5f}  std={hi_incr.std():.5f}')
print(f'  H121-Legacy   bias={bias_incr_s:+.5f}  RMSD={rmsd_incr_s:.5f}  R={r_incr_s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter
ax = axes[0]
lim = 0.06
ax.hexbin(li_incr, hi_incr, gridsize=60, cmap='viridis', mincnt=1,
          extent=[-lim, lim, -lim, lim])
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='1:1')
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('Legacy A-F increment (deg. sat.)')
ax.set_ylabel('H121 A-F increment (deg. sat.)')
ax.set_title(f'0.25 deg gridded analysis increments -- {DATE_STR}')
ax.legend(fontsize=9)
stats_lines = [f'n={both_incr.sum():,}',
               f'bias={bias_incr_s:+.5f}',
               f'RMSD={rmsd_incr_s:.5f}',
               f'R={r_incr_s:.4f}']
ax.text(0.05, 0.95, '\n'.join(stats_lines), transform=ax.transAxes, va='top',
        fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# Histogram of increments for assimilated obs
ax = axes[1]
bins_inc = np.linspace(-0.15, 0.15, 80)
leg_incr_all  = np.concatenate([ofa_all['eumetsat_assim'][s]['incr']
                                 [ofa_all['eumetsat_assim'][s]['assim'] == 1]
                                 for s in legacy_ids])
h121_incr_all = np.concatenate([ofa_all['hsaf_assim'][s]['incr']
                                 [ofa_all['hsaf_assim'][s]['assim'] == 1]
                                 for s in h121_ids])
ax.hist(leg_incr_all,  bins=bins_inc, density=True, alpha=0.6, color='steelblue',
        label=f'Legacy  mean={leg_incr_all.mean():+.4f}  std={leg_incr_all.std():.4f}')
ax.hist(h121_incr_all, bins=bins_inc, density=True, alpha=0.6, color='darkorange',
        label=f'H121    mean={h121_incr_all.mean():+.4f}  std={h121_incr_all.std():.4f}')
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('A-F increment (deg. sat., assimilated obs only)')
ax.set_ylabel('Density')
ax.set_title('Analysis increment distributions')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
